# Tennis-ball detector: test on a video

Loads the exported ONNX model, runs it frame by frame on a video at 128×128, draws the bounding box, the ball centre and the image centre, and prints the horizontal / vertical error used by the robot controller.

## 1. Environment and model

In [ ]:
!pip install ultralytics

#import yolov8 libraries
#https://docs.ultralytics.com/

import ultralytics
from ultralytics import YOLO

import cv2
import numpy as np
from google.colab.patches import cv2_imshow

# mount google colab
from google.colab import drive
drive.mount('/content/drive')

#on charge notre modèle, entrainé sur notre dataset
#tutorial : https://docs.ultralytics.com/modes/predict/#inference-sources

model = YOLO('/content/drive/MyDrive/ball_tracking/best.onnx',task='detect') # your trained model

## 2. Detection loop

In [ ]:
#on ouvre un fichier video
cap = cv2.VideoCapture('/content/drive/MyDrive/ball_tracking/your_video.mp4') #à modifier selon vos conditions

#read the video file until "q" is pressed
while( cap.isOpened() ):
    ret, frame = cap.read()
     # add resize command to fit model size
    if not ret: #si la vidéo est terminée
      break

    #resize pour correspondre à la taille du modèle
    frame = cv2.resize(frame,(128,128))

    results = model.predict(frame,imgsz=[128,128],conf=0.5)

    if len(results) > 0:
        for r in results:
            res_plotted = results[0].plot() # get results
            box = r.boxes.cpu() #get bounding boxes
            box_sz = box.xyxy.numpy() # coordinates

            if box_sz.size !=0 : # if at least one object is detected
              # Calculer le centre de la bounding box
                x_center = int((box_sz[0][0] + box_sz[0][2]) / 2)
                y_center = int((box_sz[0][1] + box_sz[0][3]) / 2)

                # Centre de l'image
                img_center_x = res_plotted.shape[1] // 2
                img_center_y = res_plotted.shape[0] // 2

                # Distance par rapport au centre (commande de suivi)
                dx = x_center - img_center_x
                dy = y_center - img_center_y

                # Dessiner le centre de la bounding box
                cv2.circle(res_plotted, (x_center, y_center), 4, (0, 0, 255), -1)

                # Dessiner le centre de l'image
                cv2.circle(res_plotted, (img_center_x, img_center_y), 4, (0, 255, 0), -1)

                # Dessiner une ligne entre les deux centres
                cv2.line(res_plotted, (x_center, y_center), (img_center_x, img_center_y), (255, 0, 0), 2)

                # Afficher les commandes de déplacement
                #cv2.putText(res_plotted, f"dx: {dx} px, dy: {dy} px", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

                # Afficher les coordonnées du centre de la balle
                #cv2.putText(res_plotted, f"Ball center: ({x_center}, {y_center})", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

                # Afficher la taille de la box
                x_min = box_sz[0][0]
                y_min = box_sz[0][1]
                x_max = box_sz[0][2]
                y_max = box_sz[0][3]
                box_X = x_max - x_min
                box_Y = y_max - y_min

                cv2_imshow(res_plotted)

                print(f"Frame - dx: {dx}, dy: {dy}, Ball: ({x_center}, {y_center})")

            break


    #c = cv2.waitKey(1)
    #if c == 27:
    #    break

cap.release()
cv2.destroyAllWindows()